In [13]:
import polars as pl
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST", "127.0.0.1"),
    port=int(os.getenv("DB_PORT", 5432)),
    database=os.getenv("DB_NAME", "phobos_records"),
    user=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASSWORD", "airvana")
)

print("Connesso al DB")

Connesso al DB


In [14]:
def query(sql):
    return pl.read_database(sql, conn)

artists = query("SELECT * FROM artists")
works = query("SELECT * FROM works")
transactions = query("SELECT * FROM transactions")
quotas = query("SELECT * FROM quotas")
royalties_by_artist = query("""
SELECT a.name, w.title, t.period, t.source,
       t.gross_rev,
       t.gross_rev - COALESCE(t.platform_fee,0) - COALESCE(t.distr_cost,0) AS net_rev,
       (t.gross_rev - COALESCE(t.platform_fee,0) - COALESCE(t.distr_cost,0)) * a.royalty_pct AS royalty
FROM transactions t
JOIN works w ON t.work_id = w.work_id
JOIN artists a ON w.artist_id = a.artist_id
ORDER BY a.name, t.period
""")


In [15]:
artist_works = artists.join(works, on="artist_id", how="inner", suffix="_work")
artist_works_transactions = artist_works.join(
    transactions, 
    on="work_id", 
    how="inner",
    suffix="_tx"
)

# Join con quotas
artist_works_quotas = artist_works.join(
    quotas, 
    on=["work_id", "artist_id"], 
    how="left",
    suffix="_quota"
)

artist_works_transactions.head()
artist_works_quotas.head()



artist_id,name,royalty_pct,advance_paid,advance_pending,is_front_artist,artist_image,main_genre,work_type,loaded_at,work_id,title,secondary_artists_id,work_cover,genre,bpm,iswc,song_key,release_date,duration,loaded_at_work,quota_id,quota_pct,loaded_at_quota
i64,str,"decimal[38,2]","decimal[38,2]","decimal[38,2]",bool,str,str,str,datetime[μs],i64,str,str,str,list[str],i64,str,str,date,i64,datetime[μs],i64,"decimal[38,2]",datetime[μs]
1,"""Lyra Void""",0.12,5000.00,2000.00,true,"""https://phobosrecords.com/imag…","""Elettronico""","""Album""",2026-03-12 20:59:04.124379,1,"""Event Horizon""","""""","""https://phobosrecords.com/cove…","[""Techno""]",126,"""T-123.456.788.Z""","""Fm""",2025-05-20,290,2026-03-12 20:59:04.124379,1,100.00,2026-03-12 20:59:04.124379
1,"""Lyra Void""",0.12,5000.00,2000.00,true,"""https://phobosrecords.com/imag…","""Elettronico""","""Album""",2026-03-12 20:59:04.124379,2,"""Dark Matter""","""""","""https://phobosrecords.com/cove…","[""Techno""]",128,"""T-123.456.789.A""","""Am""",2025-06-15,225,2026-03-12 20:59:04.124379,2,100.00,2026-03-12 20:59:04.124379
1,"""Lyra Void""",0.12,5000.00,2000.00,true,"""https://phobosrecords.com/imag…","""Elettronico""","""Album""",2026-03-12 20:59:04.124379,3,"""Quantum Dreams""","""2""","""https://phobosrecords.com/cove…","[""Ambient House""]",122,"""T-123.456.790.B""","""Fm""",2025-07-22,260,2026-03-12 20:59:04.124379,3,70.00,2026-03-12 20:59:04.124379
1,"""Lyra Void""",0.12,5000.00,2000.00,true,"""https://phobosrecords.com/imag…","""Elettronico""","""Album""",2026-03-12 20:59:04.124379,4,"""Stellar Echo""","""""","""https://phobosrecords.com/cove…","[""Trance""]",136,"""T-123.456.791.C""","""Cmaj""",2025-08-30,312,2026-03-12 20:59:04.124379,5,100.00,2026-03-12 20:59:04.124379
2,"""Solaris""",0.15,3500.00,1500.00,true,"""https://phobosrecords.com/imag…","""Ambient Techno""","""Album""",2026-03-12 20:59:04.124379,5,"""Cosmic Drift""","""1""","""https://phobosrecords.com/cove…","[""Ambient Techno""]",110,"""T-234.567.890.D""","""Dm""",2025-03-10,378,2026-03-12 20:59:04.124379,6,75.00,2026-03-12 20:59:04.124379


In [16]:
artist_works_transactions = artist_works_transactions.with_columns(
    (
        (pl.col("gross_rev") - 
         pl.col("platform_fee").fill_null(0) - 
         pl.col("distr_cost").fill_null(0)) * 
        pl.col("royalty_pct")
    ).alias("royalty_earned")
)

In [17]:
# Ora puoi fare l'aggregazione
artist_works_transactions.group_by("name").agg(
    pl.col("royalty_earned").sum().alias("total_royalty")
).sort("total_royalty", descending=True)

name,total_royalty
str,"decimal[38,2]"
"""Echo Waves""",21204.00
"""Frost Byte""",20859.00
"""Lyra Void""",20644.32
"""Velocity""",19875.20
"""Stellar Flare""",10446.80
"""Solaris""",9613.50
"""Phoenix Rise""",9020.00
"""Nebula""",7805.00
"""Crimson Tide""",6160.00


In [18]:
# Controllo quotas: somma % per work
artist_works_quotas.group_by("work_id").agg(
    pl.col("quota_pct").sum().alias("total_quota_pct")
).filter(pl.col("total_quota_pct") != 100)  # trovare errori


work_id,total_quota_pct
i64,"decimal[38,2]"
5,75.00
9,60.00
27,72.00
30,68.00
3,70.00
12,80.00
15,70.00
18,75.00
21,65.00


In [19]:
# Dettagli dei work con quota < 100%
artist_works_quotas.group_by("work_id", "title").agg(
    pl.col("quota_pct").sum().alias("total_quota_pct")
).filter(pl.col("total_quota_pct") < 100).sort("total_quota_pct")

work_id,title,total_quota_pct
i64,str,"decimal[38,2]"
9,"""Cosmic Dust""",60.00
21,"""Ancient Woods""",65.00
30,"""Digital Blizzard""",68.00
3,"""Quantum Dreams""",70.00
15,"""Storm Surge""",70.00
27,"""Bright Horizon""",72.00
18,"""Speed of Light""",75.00
5,"""Cosmic Drift""",75.00
12,"""Retro Future""",80.00
